# Notebook 1: 데이터 준비 (Zenodo10K)

## 목표
1. `Forceless/Zenodo10K`에서 PPTX 파일 스트리밍 수집 (5~50장 덱 필터링)
2. LibreOffice로 PPTX → PNG 슬라이드 이미지 변환 (224×224 letterbox)
3. python-pptx 텍스트 피처 기반 개선된 weak label 자동 생성
4. Google Drive에 저장

## stanford_slide 대비 개선
| 항목 | 기존 (stanford_slide) | 변경 (Zenodo10K) |
|------|----------------------|------------------|
| 평균 덱 길이 | 2.49장 | ~10~15장 (필터 후) |
| 슬라이드 순서 보장 | 불확실 | PPTX 원본 기준 보장 |
| Weak label 품질 | 위치만 사용 | 위치 + 텍스트 피처 |
| 역할 1·3 학습 가능 여부 | 불가 (샘플 없음) | 가능 (텍스트 피처 활용) |

## 예상 소요 시간
- PPTX 수집 + 변환: 60~90분 (600 덱 기준)
- Weak label 생성: 10분

## 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR   = '/content/drive/MyDrive/dadeum_ml'
SLIDES_DIR = f'{BASE_DIR}/slides'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

for d in [SLIDES_DIR, LABELS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print('디렉토리 구조 생성 완료')
print(f'BASE_DIR: {BASE_DIR}')

## 1. 패키지 설치

- **LibreOffice**: PPTX → PDF 변환 (Colab 기본 미설치)
- **poppler-utils / pdf2image**: PDF → PNG 변환
- **python-pptx**: 텍스트·피처 추출 (weak label용)

In [ ]:
!apt-get update -qq
!apt-get install -q -y --fix-missing libreoffice poppler-utils
!pip install -q datasets pillow tqdm pandas python-pptx pdf2image

import subprocess
result = subprocess.run(['libreoffice', '--version'], capture_output=True, text=True)
print('LibreOffice:', result.stdout.strip())
print('설치 완료')

## 2. Zenodo10K 데이터셋 구조 탐색

실제 필드 이름을 확인한 뒤 이후 셀 변수를 맞춘다.

In [ ]:
from datasets import load_dataset

print('Forceless/Zenodo10K 구조 탐색 중...')
ds_explore = load_dataset('Forceless/Zenodo10K', split='pptx', streaming=True)

first_items = []
for item in ds_explore:
    first_items.append(item)
    if len(first_items) >= 3:
        break

print(f'\n컬럼 목록: {list(first_items[0].keys())}')
print('\n각 컬럼 타입 및 샘플 값:')
for key, val in first_items[0].items():
    if isinstance(val, (bytes, bytearray)):
        print(f'  {key}: bytes (len={len(val):,})')
    else:
        print(f'  {key}: {type(val).__name__} = {repr(val)[:120]}')

In [ ]:
# ← 위 탐색 결과를 보고 실제 필드 이름이 다르면 여기서 수정
PPTX_FIELD = 'content'   # PPTX bytes 필드명
NAME_FIELD = 'filename'  # 파일명(덱 식별) 필드명

# 자동 감지 — 탐색 결과와 다를 경우 덮어씀
sample = first_items[0]
for candidate in ['content', 'file', 'pptx', 'data', 'bytes']:
    if candidate in sample and isinstance(sample[candidate], (bytes, bytearray)):
        PPTX_FIELD = candidate
        print(f'PPTX bytes 필드 자동 감지: {PPTX_FIELD}')
        break

for candidate in ['filename', 'name', 'file_name', 'title', 'id']:
    if candidate in sample:
        NAME_FIELD = candidate
        print(f'파일명 필드 자동 감지: {NAME_FIELD}')
        break

print(f'\nPPTX_FIELD = {PPTX_FIELD}')
print(f'NAME_FIELD = {NAME_FIELD}')

## 3. 덱 길이 분포 사전 조사

200개 샘플로 분포를 확인 후 `MIN_SLIDES` / `MAX_SLIDES` 필터를 결정한다.

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from pptx import Presentation
from tqdm import tqdm

PROBE_N    = 200
MIN_SLIDES = 5
MAX_SLIDES = 50

probe_lengths = []
ds_probe = load_dataset('Forceless/Zenodo10K', split='pptx', streaming=True)

for item in tqdm(ds_probe, total=PROBE_N, desc='덱 길이 조사'):
    try:
        pptx_bytes = item.get(PPTX_FIELD)
        if pptx_bytes is None:
            continue
        prs = Presentation(io.BytesIO(pptx_bytes))
        probe_lengths.append(len(prs.slides))
    except Exception:
        pass
    if len(probe_lengths) >= PROBE_N:
        break

arr       = np.array(probe_lengths)
qualified = ((arr >= MIN_SLIDES) & (arr <= MAX_SLIDES)).sum()

print(f'=== 덱 길이 분포 (n={len(probe_lengths)}) ===')
print(f'평균:     {arr.mean():.1f}장')
print(f'중앙값:   {np.median(arr):.0f}장')
print(f'범위:     {arr.min()}~{arr.max()}장')
print(f'필터 통과: {qualified}/{len(probe_lengths)} ({qualified/len(probe_lengths):.1%})')

plt.figure(figsize=(9, 4))
plt.hist(arr, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(MIN_SLIDES, color='red',    linestyle='--', label=f'min={MIN_SLIDES}')
plt.axvline(MAX_SLIDES, color='orange', linestyle='--', label=f'max={MAX_SLIDES}')
plt.xlabel('슬라이드 수 (장/덱)')
plt.ylabel('덱 수')
plt.title('Zenodo10K 덱 길이 분포 (probe 200개)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{LABELS_DIR}/deck_length_probe.png', dpi=100)
plt.show()

## 4. PPTX 수집 + 이미지 변환

`MIN_SLIDES` ~ `MAX_SLIDES` 조건을 통과한 덱만 저장.  
LibreOffice → PDF → pdf2image(poppler) 파이프라인으로 PNG 추출.  
세션 재시작 시 체크포인트에서 자동 재개.

In [ ]:
import subprocess
import tempfile
import hashlib
import json
import pickle
from pathlib import Path
from PIL import Image
from pdf2image import convert_from_path

TARGET_DECKS = 600  # 필터 통과 덱 목표 수
IMG_SIZE     = 224


def _letterbox(img: Image.Image, size: int = 224) -> Image.Image:
    w, h   = img.size
    scale  = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img    = img.resize((new_w, new_h), Image.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    canvas.paste(img, ((size - new_w) // 2, (size - new_h) // 2))
    return canvas


def pptx_to_images(pptx_bytes: bytes, out_dir: Path, deck_id: str) -> list:
    """PPTX bytes → PNG 파일 경로 리스트. LibreOffice + pdf2image 사용."""
    saved = []
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path  = Path(tmp)
        pptx_file = tmp_path / f'{deck_id}.pptx'
        pptx_file.write_bytes(pptx_bytes)

        subprocess.run(
            ['libreoffice', '--headless', '--convert-to', 'pdf',
             '--outdir', str(tmp_path), str(pptx_file)],
            capture_output=True, timeout=120
        )
        pdf_file = tmp_path / f'{deck_id}.pdf'
        if not pdf_file.exists():
            return []

        pages = convert_from_path(str(pdf_file), dpi=150)
        out_dir.mkdir(parents=True, exist_ok=True)
        for idx, page in enumerate(pages):
            out_path = out_dir / f'slide_{idx:03d}.png'
            _letterbox(page, IMG_SIZE).save(str(out_path), 'PNG', optimize=True)
            saved.append(str(out_path))
    return saved


def make_deck_id(item: dict, fallback_idx: int) -> str:
    name = item.get(NAME_FIELD, '')
    if name:
        return 'deck_' + hashlib.md5(str(name).encode()).hexdigest()[:10]
    return f'deck_{fallback_idx:06d}'


# 체크포인트 로드
checkpoint_file = Path(f'{LABELS_DIR}/zenodo_checkpoint.pkl')
if checkpoint_file.exists():
    with open(checkpoint_file, 'rb') as f:
        all_slide_paths = pickle.load(f)
    print(f'체크포인트 로드: {len(all_slide_paths)}개 덱 완료')
else:
    all_slide_paths = {}  # {deck_id: [img_path, ...]}


errors   = []
done_ids = set(all_slide_paths.keys())
ds       = load_dataset('Forceless/Zenodo10K', split='pptx', streaming=True)
pbar     = tqdm(ds, desc='PPTX 수집·변환')

for global_idx, item in enumerate(pbar):
    if len(all_slide_paths) >= TARGET_DECKS:
        break

    deck_id = make_deck_id(item, global_idx)
    if deck_id in done_ids:
        continue

    pptx_bytes = item.get(PPTX_FIELD)
    if pptx_bytes is None:
        errors.append({'deck_id': deck_id, 'reason': 'no_pptx_field'})
        continue

    # 슬라이드 수 빠른 체크
    try:
        prs      = Presentation(io.BytesIO(pptx_bytes))
        n_slides = len(prs.slides)
    except Exception as e:
        errors.append({'deck_id': deck_id, 'reason': f'parse_error: {e}'})
        continue

    if not (MIN_SLIDES <= n_slides <= MAX_SLIDES):
        continue

    # PPTX → PNG
    out_dir = Path(f'{SLIDES_DIR}/{deck_id}')
    try:
        saved = pptx_to_images(pptx_bytes, out_dir, deck_id)
    except Exception as e:
        errors.append({'deck_id': deck_id, 'reason': f'convert_error: {e}'})
        continue

    if not saved:
        errors.append({'deck_id': deck_id, 'reason': 'libreoffice_failed'})
        continue

    all_slide_paths[deck_id] = saved
    done_ids.add(deck_id)
    pbar.set_postfix({'decks': len(all_slide_paths), 'errors': len(errors)})

    if len(all_slide_paths) % 50 == 0:
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(all_slide_paths, f)

with open(checkpoint_file, 'wb') as f:
    pickle.dump(all_slide_paths, f)

total_slides = sum(len(v) for v in all_slide_paths.values())
print(f'\n=== 수집 완료 ===')
print(f'덱 수:        {len(all_slide_paths)}개')
print(f'총 슬라이드:  {total_slides}장')
print(f'평균 덱 길이: {total_slides / max(len(all_slide_paths), 1):.1f}장')
print(f'에러:         {len(errors)}건')

with open(f'{LABELS_DIR}/collect_errors.json', 'w') as f:
    json.dump(errors[:100], f, indent=2)

## 5. 수집 덱 통계 확인

In [ ]:
import pandas as pd

lengths_arr = np.array([len(v) for v in all_slide_paths.values()])

print('=== 수집 덱 통계 ===')
print(f'덱 수:       {len(all_slide_paths)}')
print(f'평균 길이:   {lengths_arr.mean():.1f}장')
print(f'중앙값:      {np.median(lengths_arr):.0f}장')
print(f'범위:        {lengths_arr.min()}~{lengths_arr.max()}장')
print(f'표준편차:    {lengths_arr.std():.1f}')

# HMM 학습 적합성 검증
total_transitions   = int((lengths_arr - 1).sum())
hmm_n_params        = (5 - 1) + 5 * (5 - 1) + 5 * (5 - 1)  # n_components=5 기준: 44
samples_per_param   = total_transitions / hmm_n_params
print(f'\n=== HMM 학습 적합성 ===')
print(f'총 전이 수:             {total_transitions:,}')
print(f'HMM 파라미터 수(5상태): {hmm_n_params}')
print(f'샘플/파라미터 비율:     {samples_per_param:.0f}:1  (권장 30:1 이상)')
print('✓ 충분' if samples_per_param >= 30 else '⚠ 부족 — TARGET_DECKS 늘리기 권장')

plt.figure(figsize=(9, 4))
plt.hist(lengths_arr, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(lengths_arr.mean(), color='red', linestyle='--',
            label=f'평균 {lengths_arr.mean():.1f}장')
plt.xlabel('슬라이드 수 (장/덱)')
plt.ylabel('덱 수')
plt.title('Zenodo10K 수집 덱 길이 분포')
plt.legend()
plt.tight_layout()
plt.savefig(f'{LABELS_DIR}/collected_deck_dist.png', dpi=100)
plt.show()

## 6. 개선된 Weak Label 생성

PPTX 원본에서 텍스트·시각 피처를 추출해 위치 기반보다 정교한 라벨을 부여한다.

| 역할 | 우선 규칙 |
|------|----------|
| 0: 표지 | 첫 슬라이드 (무조건) |
| 1: 섹션헤더 | 텍스트 < 60자 AND 최대 폰트 ≥ 24pt AND 중간 위치 |
| 2: 본문 | 기본값 (중간 슬라이드) |
| 3: 도표/시각자료 | 차트·표 보유 OR (이미지 보유 AND 텍스트 < 80자) |
| 4: 마무리 | 마지막 슬라이드 (무조건) |

In [ ]:
from pptx.enum.shapes import MSO_SHAPE_TYPE

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']


def extract_slide_features(slide) -> dict:
    text_chunks = []
    max_font_pt = 0.0
    has_chart   = False
    has_table   = False
    has_image   = False

    for shape in slide.shapes:
        stype = shape.shape_type
        if stype == MSO_SHAPE_TYPE.CHART:
            has_chart = True
        elif stype == MSO_SHAPE_TYPE.TABLE:
            has_table = True
        elif stype in (MSO_SHAPE_TYPE.PICTURE, MSO_SHAPE_TYPE.LINKED_PICTURE):
            has_image = True

        if shape.has_text_frame:
            for para in shape.text_frame.paragraphs:
                for run in para.runs:
                    text_chunks.append(run.text.strip())
                    try:
                        sz = run.font.size
                        if sz:
                            max_font_pt = max(max_font_pt, sz / 12700)
                    except Exception:
                        pass

    full_text = ' '.join(t for t in text_chunks if t)
    return {
        'word_count':  len(full_text.split()),
        'char_count':  len(full_text),
        'max_font_pt': round(max_font_pt, 1),
        'has_chart':   has_chart,
        'has_table':   has_table,
        'has_image':   has_image,
    }


def assign_weak_label(rank: int, total: int, feats: dict) -> int:
    if rank == 0:
        return 0
    if rank == total - 1:
        return 4
    if feats['has_chart'] or feats['has_table']:
        return 3
    if feats['has_image'] and feats['char_count'] < 80:
        return 3
    if feats['char_count'] < 60 and feats['max_font_pt'] >= 24:
        return 1
    return 2


print('Weak label 생성 중 (PPTX 재파싱)...')
records      = []
processed_ids = set()
ds2 = load_dataset('Forceless/Zenodo10K', split='pptx', streaming=True)

for global_idx, item in enumerate(tqdm(ds2, desc='라벨 생성')):
    if len(processed_ids) >= len(all_slide_paths):
        break

    deck_id = make_deck_id(item, global_idx)
    if deck_id not in all_slide_paths or deck_id in processed_ids:
        continue

    pptx_bytes = item.get(PPTX_FIELD)
    if pptx_bytes is None:
        continue

    try:
        prs       = Presentation(io.BytesIO(pptx_bytes))
        n_slides  = len(prs.slides)
        img_paths = all_slide_paths[deck_id]

        for rank, slide in enumerate(prs.slides):
            if rank >= len(img_paths):
                break
            feats = extract_slide_features(slide)
            label = assign_weak_label(rank, n_slides, feats)
            records.append({
                'deck_id':        deck_id,
                'slide_idx':      rank,
                'total_slides':   n_slides,
                'position_ratio': round(rank / max(n_slides - 1, 1), 4),
                'word_count':     feats['word_count'],
                'char_count':     feats['char_count'],
                'max_font_pt':    feats['max_font_pt'],
                'has_chart':      feats['has_chart'],
                'has_table':      feats['has_table'],
                'has_image':      feats['has_image'],
                'weak_label':     label,
                'role_name':      ROLE_NAMES[label],
                'image_path':     img_paths[rank],
            })
        processed_ids.add(deck_id)
    except Exception as e:
        print(f'  ⚠ {deck_id}: {e}')

df = pd.DataFrame(records)
print(f'\n총 슬라이드: {len(df)}장 ({df["deck_id"].nunique()}개 덱)')
print(df['role_name'].value_counts())

## 7. Weak Label 품질 체크

섹션헤더(1)·도표(3)가 5% 미만이면 임계값을 완화하거나  
Notebook 06 CLIP 라벨링으로 보완한다.

In [ ]:
print('=== 클래스 분포 ===')
for role_id in range(5):
    n     = (df['weak_label'] == role_id).sum()
    ratio = n / len(df)
    flag  = ' ⚠ 소수 클래스 (<5%)' if ratio < 0.05 else ''
    print(f'  {ROLE_NAMES[role_id]:12s}: {n:5d}장 ({ratio:.1%}){flag}')

section_ratio = (df['weak_label'] == 1).sum() / len(df)
visual_ratio  = (df['weak_label'] == 3).sum() / len(df)
if section_ratio < 0.05:
    print('\n⚠ 섹션헤더 비율 낮음 → assign_weak_label: char_count < 60 → < 80 완화 고려')
if visual_ratio < 0.05:
    print('⚠ 도표 비율 낮음 → Notebook 06 CLIP 라벨링으로 보완 예정')

print('\n=== 시퀀스 역할 다양성 ===')
roles_per_deck = df.groupby('deck_id')['weak_label'].nunique()
print(f'평균 역할 다양성:       {roles_per_deck.mean():.1f}가지/덱')
print(f'3가지 이상 역할 포함:   {(roles_per_deck >= 3).sum()}개 ({(roles_per_deck >= 3).mean():.1%})')
if (roles_per_deck >= 3).mean() < 0.5:
    print('⚠ 절반 이상 덱이 2가지 역할 이하 — HMM 학습 신호 약함. 임계값 재조정 필요.')

## 8. 클래스 가중치 저장

In [ ]:
class_counts = df['weak_label'].value_counts().sort_index()

weights = {
    str(i): float(1.0 / class_counts[i])
            if i in class_counts.index and class_counts[i] > 0
            else 0.0
    for i in range(5)
}

with open(f'{LABELS_DIR}/class_weights.json', 'w') as f:
    json.dump(weights, f, indent=2)

df.to_csv(f'{LABELS_DIR}/weak_labels.csv', index=False)
print(f'저장 완료: {LABELS_DIR}/weak_labels.csv')
print(f'클래스 가중치: {weights}')

## 9. 샘플 시각화

In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for role_id, ax in enumerate(axes):
    subset = df[df['weak_label'] == role_id]
    if len(subset) == 0:
        ax.set_title(f'{ROLE_NAMES[role_id]}\n(샘플 없음)')
        ax.axis('off')
        continue
    sample = subset.sample(1).iloc[0]
    p = Path(sample['image_path'])
    if p.exists():
        ax.imshow(mpimg.imread(str(p)))
    ax.set_title(f'{ROLE_NAMES[role_id]}\n({len(subset)}장)', fontsize=10)
    ax.axis('off')

plt.suptitle('역할별 슬라이드 샘플 (Zenodo10K)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{LABELS_DIR}/role_samples.png', dpi=100)
plt.show()
print('시각화 저장 완료')

## 10. HMM 학습용 시퀀스 데이터 생성

`MIN_SLIDES` 이상인 덱만 포함 — HMM 샘플/파라미터 비율을 재확인한다.

In [ ]:
from collections import Counter

sequences = []
for deck_id, group in df.groupby('deck_id'):
    group = group.sort_values('slide_idx')
    seq   = group['weak_label'].tolist()
    if len(seq) >= MIN_SLIDES:
        sequences.append({'deck_id': deck_id, 'sequence': seq, 'length': len(seq)})

seq_df = pd.DataFrame(sequences)
seq_df.to_csv(f'{LABELS_DIR}/sequences.csv', index=False)

total_trans         = int((seq_df['length'] - 1).sum())
samples_per_param   = total_trans / 44

print(f'시퀀스 수:              {len(seq_df)}개')
print(f'평균 덱 길이:           {seq_df["length"].mean():.1f}장')
print(f'최대 덱 길이:           {seq_df["length"].max()}장')
print(f'총 전이 수:             {total_trans:,}')
print(f'샘플/파라미터 비율:     {samples_per_param:.0f}:1')
print('✓ HMM 학습 충분' if samples_per_param >= 30 else '⚠ 부족 (권장 30:1 이상)')

print('\n가장 흔한 시퀀스 패턴 (12장 이하):')
short_seqs = seq_df[seq_df['length'] <= 12]['sequence'].tolist()
for pattern, count in Counter([tuple(s) for s in short_seqs]).most_common(5):
    readable = ' → '.join([ROLE_NAMES[r] for r in pattern])
    print(f'  {readable}  ({count}개)')

In [ ]:
print('=== Notebook 1 완료 ===')
print(f'데이터셋:         Forceless/Zenodo10K')
print(f'슬라이드 이미지:  {len(df)}장 ({df["deck_id"].nunique()}개 덱)')
print(f'평균 덱 길이:     {df.groupby("deck_id").size().mean():.1f}장')
print(f'약한 라벨 CSV:    {LABELS_DIR}/weak_labels.csv')
print(f'클래스 가중치:    {LABELS_DIR}/class_weights.json')
print(f'시퀀스 CSV:       {LABELS_DIR}/sequences.csv')
print(f'체크포인트:       {LABELS_DIR}/zenodo_checkpoint.pkl')
print('\nNotebook 2 (CNN 역할 분류기)로 이동하세요.')